# Ethiopia Climate EDA
Task 2 notebook template aligned to the Week 0 rubric.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

In [ ]:
country = 'Ethiopia'
df = pd.read_csv('../data/ethiopia.csv')
df['Country'] = country
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j', errors='coerce')
df['Month'] = df['DATE'].dt.month
df['YearMonth'] = df['DATE'].dt.to_period('M').dt.to_timestamp()
df.head()

## Data Profiling & Cleaning
- Replace `-999` with `NaN`
- Report and remove duplicates
- Summarize missingness and outliers
- Apply forward-fill for weather columns and drop rows with >30% missing

In [ ]:
df = df.replace(-999, np.nan)
dup_count = df.duplicated().sum()
print('Duplicate rows:', dup_count)
df = df.drop_duplicates().copy()
display(df.describe(include='all'))
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df)) * 100
display(missing.sort_values('missing_pct', ascending=False))

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_df = df[z_cols].apply(lambda s: zscore(s, nan_policy='omit'))
outliers = df[z_df.abs().gt(3).any(axis=1)]
print('Outlier rows flagged (|Z|>3):', len(outliers))
row_missing_ratio = df.isna().mean(axis=1)
df = df.loc[row_missing_ratio <= 0.30].copy()
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']
df[weather_cols] = df[weather_cols].ffill()

In [ ]:
monthly_t2m = df.groupby('YearMonth', as_index=False)['T2M'].mean()
monthly_precip = df.groupby('YearMonth', as_index=False)['PRECTOTCORR'].sum()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly_t2m['YearMonth'], monthly_t2m['T2M'])
ax.set_title('Monthly Average T2M (Ethiopia)')
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly_precip['YearMonth'], monthly_precip['PRECTOTCORR'])
ax.set_title('Monthly Total PRECTOTCORR (Ethiopia)')
ax.set_xlabel('Month')
ax.set_ylabel('Precipitation (mm)')
plt.show()

In [ ]:
numeric = df.select_dtypes(include='number')
plt.figure(figsize=(10, 8))
sns.heatmap(numeric.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0], alpha=0.5)
axes[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1], alpha=0.5)
axes[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['PRECTOTCORR'].dropna(), kde=True, ax=ax)
ax.set_title('PRECTOTCORR Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['T2M'], df['RH2M'], s=df['PRECTOTCORR'].fillna(0) * 3 + 5, alpha=0.3)
plt.xlabel('T2M')
plt.ylabel('RH2M')
plt.title('Bubble Chart: T2M vs RH2M (size=PRECTOTCORR)')
plt.show()

In [ ]:
from pathlib import Path
Path('../data').mkdir(exist_ok=True)
df.to_csv('../data/ethiopia_clean.csv', index=False)
print('Saved: ../data/ethiopia_clean.csv')